In [1]:
from pathlib import Path
import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

con = duckdb.connect()

transaction_files = [
    "transactions.csv",
    "transactions_v2.csv",
]

for name in transaction_files:
    path = RAW_DIR / name
    print(f"{name}: exists={path.exists()}")

transactions.csv: exists=True
transactions_v2.csv: exists=True


In [2]:
for name in transaction_files:
    preview = pd.read_csv(RAW_DIR / name, nrows=5)
    print(name)
    print("Columns:", preview.columns.tolist())
    display(preview)

transactions.csv
Columns: ['msno', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'transaction_date', 'membership_expire_date', 'is_cancel']


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,YyO+tlZtAXYXoZhNr3Vg3+dfVQvrBVGO8j1mfqe4ZHc=,41,30,129,129,1,20150930,20151101,0
1,AZtu6Wl0gPojrEQYB8Q3vBSmE2wnZ3hi1FbK1rQQ0A4=,41,30,149,149,1,20150930,20151031,0
2,UkDFI97Qb6+s2LWcijVVv4rMAsORbVDT2wNXF0aVbns=,41,30,129,129,1,20150930,20160427,0
3,M1C56ijxozNaGD0t2h68PnH2xtx5iO5iR2MVYQB6nBI=,39,30,149,149,1,20150930,20151128,0
4,yvj6zyBUaqdbUQSrKsrZ+xNDVM62knauSZJzakS9OW4=,39,30,149,149,1,20150930,20151121,0


transactions_v2.csv
Columns: ['msno', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'transaction_date', 'membership_expire_date', 'is_cancel']


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,++6eU4LsQ3UQ20ILS7d99XK8WbiVgbyYL4FUgzZR134=,32,90,298,298,0,20170131,20170504,0
1,++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOeA=,41,30,149,149,1,20150809,20190412,0
2,+/GXNtXWQVfKrEDqYAzcSw2xSPYMKWNj22m+5XkVQZc=,36,30,180,180,1,20170303,20170422,0
3,+/w1UrZwyka4C9oNH3+Q8fUf3fD8R3EwWrx57ODIsqk=,36,30,180,180,1,20170329,20170331,1
4,+00PGzKTYqtnb65mPKPyeHXcZEwqiEzktpQksaaSC3c=,41,30,99,99,1,20170323,20170423,0


In [3]:
date_audit_sql = """
WITH parsed AS (
    SELECT
        msno,
        TRY_STRPTIME(
            transaction_date, '%Y%m%d'
        )::DATE AS transaction_day,
        TRY_STRPTIME(
            membership_expire_date, '%Y%m%d'
        )::DATE AS expire_day
    FROM read_csv(?, all_varchar = true)
)
SELECT
    COUNT(*) AS rows,
    COUNT(*) FILTER (
        WHERE msno IS NULL OR TRIM(msno) = ''
    ) AS missing_user_ids,
    MIN(transaction_day) AS first_transaction,
    MAX(transaction_day) AS last_transaction,
    COUNT(*) FILTER (
        WHERE transaction_day IS NULL
    ) AS missing_or_invalid_transaction_dates,
    MIN(expire_day) AS earliest_expiration,
    MAX(expire_day) AS latest_expiration,
    COUNT(*) FILTER (
        WHERE expire_day IS NULL
    ) AS missing_or_invalid_expiration_dates
FROM parsed
"""

audit_results = []

for name in transaction_files:
    print(f"Checking {name}...", flush=True)

    result = con.execute(
        date_audit_sql,
        [str(RAW_DIR / name)],
    ).df()

    result.insert(0, "file", name)
    audit_results.append(result)

transaction_date_audit = pd.concat(
    audit_results,
    ignore_index=True,
)

transaction_date_audit

Checking transactions.csv...
Checking transactions_v2.csv...


,file,rows,missing_user_ids,first_transaction,last_transaction,missing_or_invalid_transaction_dates,earliest_expiration,latest_expiration,missing_or_invalid_expiration_dates
0,transactions.csv,21547746,0,2015-01-01,2017-02-28,0,1970-01-01,2017-03-31,0
1,transactions_v2.csv,1431009,0,2015-01-01,2017-03-31,0,2016-04-19,2036-10-15,0


In [4]:
monthly_sql = """
SELECT
    STRFTIME(
        TRY_STRPTIME(transaction_date, '%Y%m%d'),
        '%Y-%m'
    ) AS transaction_month,
    COUNT(*) AS transactions
FROM read_csv(?, all_varchar = true)
GROUP BY transaction_month
ORDER BY transaction_month
"""

monthly_results = []

for name in transaction_files:
    result = con.execute(
        monthly_sql,
        [str(RAW_DIR / name)],
    ).df()

    result.insert(0, "file", name)
    monthly_results.append(result)

transaction_monthly = pd.concat(
    monthly_results,
    ignore_index=True,
)

for name in transaction_files:
    print(name)
    display(
        transaction_monthly.loc[
            transaction_monthly["file"] == name,
            ["transaction_month", "transactions"],
        ]
    )

transactions.csv


,transaction_month,transactions
0,2015-01,576349
1,2015-02,569421
2,2015-03,657379
3,2015-04,595444
4,2015-05,601063
5,2015-06,818903
6,2015-07,697086
7,2015-08,740477
8,2015-09,744636
9,2015-10,716146


transactions_v2.csv


,transaction_month,transactions
26,2015-01,1169
27,2015-02,1504
28,2015-03,3371
29,2015-04,3579
30,2015-05,4393
31,2015-06,5179
32,2015-07,6247
33,2015-08,7095
34,2015-09,7692
35,2015-10,9222


In [5]:
v2_monthly = transaction_monthly.loc[
    transaction_monthly["file"] == "transactions_v2.csv"
].copy()

v2_history_rows = v2_monthly.loc[
    v2_monthly["transaction_month"] < "2017-03",
    "transactions",
].sum()

v2_march_rows = v2_monthly.loc[
    v2_monthly["transaction_month"] == "2017-03",
    "transactions",
].sum()

print(f"V2 historical transactions: {v2_history_rows:,}")
print(f"V2 March transactions: {v2_march_rows:,}")

V2 historical transactions: 361,187
V2 March transactions: 1,069,822


In [6]:
v1_columns = pd.read_csv(
    RAW_DIR / "transactions.csv", nrows=0
).columns.tolist()

v2_columns = pd.read_csv(
    RAW_DIR / "transactions_v2.csv", nrows=0
).columns.tolist()

assert set(v1_columns) == set(v2_columns), (
    "The transaction files have different columns."
)

print("Fields to compare:", v1_columns)

Fields to compare: ['msno', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'transaction_date', 'membership_expire_date', 'is_cancel']


In [7]:
# Quote SQL column names and escape embedded double quotes.
quoted_columns = [
    '"' + column.replace('"', '""') + '"'
    for column in v1_columns
]

matching_conditions = " AND ".join(
    f"v1.{column} IS NOT DISTINCT FROM v2.{column}"
    for column in quoted_columns
)

overlap_sql = f"""
WITH v1 AS (
    SELECT *
    FROM read_csv(?, all_varchar = true)
),
v2_history AS (
    SELECT *
    FROM read_csv(?, all_varchar = true)
    WHERE TRY_STRPTIME(transaction_date, '%Y%m%d')::DATE
          < DATE '2017-03-01'
),
checked AS (
    SELECT
        EXISTS (
            SELECT 1
            FROM v1
            WHERE {matching_conditions}
        ) AS exact_match
    FROM v2_history AS v2
)
SELECT
    COUNT(*) AS v2_history_rows,
    COUNT(*) FILTER (
        WHERE exact_match
    ) AS rows_found_in_v1,
    COUNT(*) FILTER (
        WHERE NOT exact_match
    ) AS rows_not_found_in_v1
FROM checked
"""

history_overlap = con.execute(
    overlap_sql,
    [
        str(RAW_DIR / "transactions.csv"),
        str(RAW_DIR / "transactions_v2.csv"),
    ],
).df()

history_overlap

,v2_history_rows,rows_found_in_v1,rows_not_found_in_v1
0,361187,0,361187


In [8]:
# Compare all fields except membership_expire_date.
comparison_columns = [
    column
    for column in v1_columns
    if column != "membership_expire_date"
]

quoted_comparison_columns = [
    '"' + column.replace('"', '""') + '"'
    for column in comparison_columns
]

core_matching_conditions = " AND ".join(
    f"v1.{column} IS NOT DISTINCT FROM v2.{column}"
    for column in quoted_comparison_columns
)

comparison_sql = f"""
WITH v1 AS (
    SELECT *
    FROM read_csv(?, all_varchar = true)
),
v2_history AS (
    SELECT *
    FROM read_csv(?, all_varchar = true)
    WHERE TRY_STRPTIME(transaction_date, '%Y%m%d')::DATE
          < DATE '2017-03-01'
),
checked AS (
    SELECT
        EXISTS (
            SELECT 1
            FROM v1
            WHERE v1.msno = v2.msno
              AND v1.transaction_date = v2.transaction_date
        ) AS same_user_and_date,
        EXISTS (
            SELECT 1
            FROM v1
            WHERE {core_matching_conditions}
        ) AS same_except_expiration
    FROM v2_history AS v2
)
SELECT
    COUNT(*) AS v2_history_rows,
    COUNT(*) FILTER (
        WHERE same_user_and_date
    ) AS matching_user_and_date,
    COUNT(*) FILTER (
        WHERE same_except_expiration
    ) AS matching_except_expiration
FROM checked
"""

history_comparison = con.execute(
    comparison_sql,
    [
        str(RAW_DIR / "transactions.csv"),
        str(RAW_DIR / "transactions_v2.csv"),
    ],
).df()

history_comparison

,v2_history_rows,matching_user_and_date,matching_except_expiration
0,361187,9967,6126


In [9]:
v2_duplicate_audit = con.execute(
    """
    WITH duplicate_groups AS (
        SELECT
            *,
            COUNT(*) AS occurrences
        FROM read_csv(?, all_varchar = true)
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS duplicate_groups,
        COALESCE(SUM(occurrences), 0) AS rows_in_duplicate_groups,
        COALESCE(SUM(occurrences - 1), 0) AS excess_duplicate_rows
    FROM duplicate_groups
    """,
    [str(RAW_DIR / "transactions_v2.csv")],
).df()

v2_duplicate_audit

,duplicate_groups,rows_in_duplicate_groups,excess_duplicate_rows
0,0,0.0,0.0


In [10]:
quality_sql = """
WITH parsed AS (
    SELECT
        TRY_CAST(payment_plan_days AS INTEGER) AS plan_days,
        TRY_CAST(plan_list_price AS DOUBLE) AS list_price,
        TRY_CAST(actual_amount_paid AS DOUBLE) AS amount_paid,
        TRY_CAST(is_auto_renew AS INTEGER) AS auto_renew,
        TRY_CAST(is_cancel AS INTEGER) AS cancel_flag
    FROM read_csv(?, all_varchar = true)
)
SELECT
    COUNT(*) AS rows,
    COUNT(*) FILTER (
        WHERE plan_days IS NULL
    ) AS missing_or_invalid_plan_days,
    COUNT(*) FILTER (
        WHERE plan_days < 0
    ) AS negative_plan_days,
    COUNT(*) FILTER (
        WHERE plan_days = 0
    ) AS zero_plan_days,
    COUNT(*) FILTER (
        WHERE list_price IS NULL OR amount_paid IS NULL
    ) AS missing_or_invalid_amounts,
    COUNT(*) FILTER (
        WHERE list_price < 0 OR amount_paid < 0
    ) AS negative_amount_rows,
    COUNT(*) FILTER (
        WHERE amount_paid = 0
    ) AS zero_payment_rows,
    COUNT(*) FILTER (
        WHERE auto_renew IS NULL OR auto_renew NOT IN (0, 1)
    ) AS invalid_auto_renew,
    COUNT(*) FILTER (
        WHERE cancel_flag IS NULL OR cancel_flag NOT IN (0, 1)
    ) AS invalid_cancel,
    MAX(plan_days) AS max_plan_days,
    MAX(amount_paid) AS max_amount_paid
FROM parsed
"""

quality_results = []

for name in transaction_files:
    print(f"Checking {name}...", flush=True)

    result = con.execute(
        quality_sql,
        [str(RAW_DIR / name)],
    ).df()

    result.insert(0, "file", name)
    quality_results.append(result)

transaction_quality = pd.concat(
    quality_results,
    ignore_index=True,
)

display(transaction_quality.T)

Checking transactions.csv...
Checking transactions_v2.csv...


,0,1
file,transactions.csv,transactions_v2.csv
rows,21547746,1431009
missing_or_invalid_plan_days,0,0
negative_plan_days,0,0
zero_plan_days,870124,2218
missing_or_invalid_amounts,0,0
negative_amount_rows,0,0
zero_payment_rows,1196876,21448
invalid_auto_renew,0,0
invalid_cancel,0,0


In [11]:
report_dir = PROJECT_ROOT / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

audit_tables = {
    "transaction_date_audit": transaction_date_audit,
    "transaction_monthly": transaction_monthly,
    "transaction_history_overlap": history_overlap,
    "transaction_history_comparison": history_comparison,
    "transaction_v2_duplicates": v2_duplicate_audit,
    "transaction_quality": transaction_quality,
}

for name, table in audit_tables.items():
    table.to_csv(report_dir / f"{name}.csv", index=False)

print("Transaction audit reports saved.")

Transaction audit reports saved.
